# 🎯 HW3 — Clustering
## Configuring the Night-Shift Waste Machines

> **The big idea:** you can't hand-tune a machine for every single waste batch — but you *can* sort
> the batches into a handful of **groups** and run one well-tuned machine per group.

You manage a nuclear power plant. Every night a fleet of **waste-processing machines** runs through
the day's radioactive waste. Each machine is **set to specific settings** (shielding, temperature,
throughput…) that must match the waste it processes. Tonight you receive a **log of waste batches** —
hundreds of them, each with physical properties and a quantity.

**The trade-off** ⚖️
- Using **more machines** means each one is tuned tightly to its waste → accurate, but expensive.
- Using **fewer machines** is cheaper, but each one is a compromise → some waste is badly processed.

Your job: find the **smallest number of machines** that still keeps every batch well-matched to a
machine. That is exactly an **unsupervised clustering** problem — one cluster ⇒ one machine setting,
and the **optimal number of clusters ⇒ the optimal number of machines**.

**The workflow you'll build**
```
raw waste log → visualize → select features → preprocess → cluster (find k) → machine settings
```

This notebook runs top-to-bottom on **Google Colab**. Cells marked **🎯 TASK** are for you to
complete; visualization code is hidden behind a title (double-click to peek).


## Part 0 — Setup

This notebook is **self-contained**: when you run the first cell it pulls the rest of the
exercise files (the `hw3_data.py` helper and the waste dataset under `data/`) directly from the
course repository. Run the setup cells below in order.

> ✅ The course repo is **public**, so no GitHub token is needed — the cell clones it directly.
> If you're running this notebook from inside a local clone of the repo, the setup cell skips the
> clone and just uses the files already on disk.

**0.1 — Fetch the exercise files.**

In [ ]:
import os, sys

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE0-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"

# Locate this homework folder (holds helpers/ + data/) and make the helpers importable.
_HW = os.path.join("workshop", "homework")
for _root in [os.path.join(REPO_NAME, _HW), ".", _HW,
              os.path.dirname(os.getcwd()), os.getcwd()]:
    if os.path.exists(os.path.join(_root, "helpers", "hw3_data.py")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        "Could not find the homework folder (workshop/homework with helpers/ + data/). "
        "On Colab, re-run this cell to clone it; locally, run the notebook from inside a clone of the FDD-WE0-public repo.")
sys.path.insert(0, os.path.join(os.getcwd(), "helpers"))   # make the helpers importable
print("Working directory:", os.getcwd())


**0.2 — Install dependencies.** All of these are already on Colab; this just guarantees
the right versions (and makes the notebook work outside Colab too).

In [ ]:
%pip install -q -r requirements_block2.txt

**0.3 — Import the libraries** we'll use throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (silhouette_score, r2_score, mean_absolute_error,
                             accuracy_score, precision_score, recall_score, confusion_matrix)

import sklearn
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Shared colour palette (used throughout).
BLUE, ORANGE, GREEN, RED = "#6f7bf0", "#dd8452", "#55a868", "#c44e52"
CLUSTER_COLORS = [BLUE, ORANGE, GREEN, RED, "#9a6fe2", "#c56fbe", "#5bc0be", "#e0b13a"]
print("scikit-learn", sklearn.__version__, "ready ✅")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
def workflow_diagram():
    steps = [("🗂️", "Waste log", "raw batches"), ("📊", "Visualize", "see the data"),
             ("🧹", "Select", "useful features"), ("⚖️", "Preprocess", "log + scale"),
             ("🧩", "Cluster", "find k machines"), ("🛠️", "Settings", "tune each machine"),
             ("🚚", "Route", "new arrivals"), ("🔮", "Forecast", "fill gaps + re-plan")]
    grad = ["#667eea", "#6f7bf0", "#55a868", "#3fa7a0", "#dd8452", "#c56fbe", "#5b8def", "#d36bb0"]
    blocks = ""
    for (ic, t, d), g in zip(steps, grad):
        blocks += (f'<div class="pl-step"><div class="pl-ic" style="background:linear-gradient(135deg,{g},{g}cc)">{ic}</div>'
                   f'<div class="pl-t">{t}</div><div class="pl-d">{d}</div></div>'
                   '<div class="pl-ar">➜</div>')
    blocks = blocks.rsplit('<div class="pl-ar">➜</div>', 1)[0]
    display(HTML(f'''
<style>
.pl{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f6f8ff,#f3fbf6);
    border-radius:18px;padding:20px 16px;margin:8px 0;border:1px solid #e8efea}}
.pl-h{{font-size:20px;font-weight:800;color:#2d4b3b;margin:0 0 14px}}
.pl-row{{display:flex;align-items:flex-start;flex-wrap:wrap;gap:0}}
.pl-step{{flex:1 1 96px;min-width:96px;text-align:center;padding:0 2px}}
.pl-ic{{width:48px;height:48px;border-radius:50%;margin:0 auto 7px;display:flex;align-items:center;
       justify-content:center;font-size:21px;color:#fff;box-shadow:0 6px 14px rgba(85,168,104,.35)}}
.pl-t{{font-weight:700;font-size:12px;color:#23502c}}
.pl-d{{font-size:10px;color:#7f9b86;margin-top:2px}}
.pl-ar{{display:flex;align-items:center;font-size:18px;color:#a9d6ba;flex:0 0 14px;height:48px}}
</style>
<div class="pl"><div class="pl-h">🌙 The night-shift workflow</div><div class="pl-row">{blocks}</div></div>'''))

workflow_diagram()

### The core idea — in one picture

Run the cell below for a visual of what you're about to do: a messy pile of waste batches gets
**sorted into a few groups**, and each group gets **one machine** tuned to it. The slider underneath
shows the trade-off you're balancing.

> ⚠️ **Why the grouping has to be good:** a machine can **only process waste that matches its
> settings**. Feed it a batch from a different group and it's mis-handled — under-shielded, wrong
> temperature, unsafe. So every batch *must* end up with a machine whose settings fit it. That's the
> whole point of clustering well: tight, clean groups mean every batch is sent to a machine that can
> actually process it.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
def concept_diagram():
    rng = np.random.default_rng(7)
    PAL = [BLUE, ORANGE, GREEN, RED, "#9a6fe2"]
    centers = [(62, 56), (148, 50), (104, 108), (56, 150), (158, 140)]
    pts = []
    for ci, (cx, cy) in enumerate(centers):
        for _ in range(8):
            pts.append((cx + rng.normal(0, 13), cy + rng.normal(0, 13), ci))

    grey = "".join(f'<circle cx="{x:.0f}" cy="{y:.0f}" r="4.5" fill="#c3c8de"/>' for x, y, _ in pts)
    color = "".join(f'<circle cx="{x:.0f}" cy="{y:.0f}" r="4.5" fill="{PAL[c]}"/>' for x, y, c in pts)
    halos = "".join(
        f'<circle cx="{cx}" cy="{cy}" r="32" fill="{PAL[i]}" opacity="0.10"/>'
        f'<circle cx="{cx}" cy="{cy}" r="32" fill="none" stroke="{PAL[i]}" '
        f'stroke-dasharray="4 3" opacity="0.55"/>' for i, (cx, cy) in enumerate(centers))
    chips = "".join(
        f'<div class="cd-m" style="border-left-color:{PAL[i]}">'
        f'<span style="font-size:18px">🛠️</span>'
        f'<div><div class="cd-mt">Machine {i + 1}</div>'
        f'<div class="cd-md">only accepts <b style="color:{PAL[i]}">group {i + 1}</b></div></div></div>'
        for i in range(len(centers)))

    def panel(title, svg_inner):
        return (f'<div class="cd-panel"><div class="cd-cap">{title}</div>'
                f'<svg viewBox="0 0 210 200" width="100%" height="190">{svg_inner}</svg></div>')

    p1 = panel("① Today's waste log<br><span>hundreds of mixed batches</span>", grey)
    p2 = panel("② Cluster them<br><span>find natural groups (k=?)</span>", halos + color)
    p3 = (f'<div class="cd-panel"><div class="cd-cap">③ One machine per group'
          f'<br><span>each tuned to its waste</span></div><div class="cd-mwrap">{chips}</div>'
          f'<div class="cd-rule">⚠️ a machine <b>only</b> processes waste matching its settings</div></div>')
    arrow = '<div class="cd-ar">➜</div>'

    display(HTML(f'''
<style>
.cd{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f7f9ff,#f3fbf6);
    border:1px solid #e7eef0;border-radius:20px;padding:20px 18px;margin:8px 0}}
.cd-row{{display:flex;align-items:stretch;gap:6px;flex-wrap:wrap;justify-content:center}}
.cd-panel{{flex:1 1 220px;min-width:200px;background:#fff;border:1px solid #eef1f7;border-radius:14px;
    padding:12px 10px;box-shadow:0 4px 12px rgba(50,70,120,.05)}}
.cd-cap{{font-weight:800;font-size:13px;color:#2c2350;text-align:center;line-height:1.45;margin-bottom:4px}}
.cd-cap span{{font-weight:500;font-size:11px;color:#8b86a6}}
.cd-ar{{display:flex;align-items:center;font-size:26px;color:#b7c3d6;flex:0 0 22px}}
.cd-mwrap{{display:flex;flex-direction:column;gap:7px;padding:8px 4px 2px}}
.cd-m{{display:flex;align-items:center;gap:10px;background:#fafbff;border:1px solid #eef1f7;
    border-left:6px solid;border-radius:9px;padding:7px 11px}}
.cd-mt{{font-weight:700;font-size:12.5px;color:#2c2350}}
.cd-md{{font-size:11px;color:#8b86a6}}
.cd-rule{{margin:9px 4px 2px;padding:7px 9px;background:#fff7ed;border:1px solid #f4d9bd;
    border-radius:9px;font-size:11px;color:#8a4b1e;line-height:1.4}}
.cd-trade{{margin-top:16px;background:#fff;border:1px solid #eef1f7;border-radius:14px;padding:14px 16px}}
.cd-th{{font-weight:800;font-size:13px;color:#2c2350;margin-bottom:10px}}
.cd-bar{{position:relative;height:16px;border-radius:10px;
    background:linear-gradient(90deg,#55a868,#e7c14a 50%,#c44e52)}}
.cd-pin{{position:absolute;top:-7px;left:50%;transform:translateX(-50%);font-size:20px}}
.cd-ends{{display:flex;justify-content:space-between;margin-top:8px;font-size:11.5px}}
.cd-end{{flex:1 1 0;max-width:46%}}
.cd-end b{{display:block;font-size:12.5px;color:#2c2350}}
.cd-end.l{{text-align:left}} .cd-end.r{{text-align:right}}
.cd-end .sub{{color:#8b86a6}}
.cd-sweet{{text-align:center;font-size:12px;color:#1f9d55;font-weight:700;margin-top:6px}}
</style>
<div class="cd">
  <div class="cd-row">{p1}{arrow}{p2}{arrow}{p3}</div>
  <div class="cd-trade">
    <div class="cd-th">⚖️ How many machines? — the trade-off you'll optimise</div>
    <div class="cd-bar"><div class="cd-pin">📍</div></div>
    <div class="cd-ends">
      <div class="cd-end l"><b>Too few machines</b><span class="sub">cheap, but each is a rough compromise</span></div>
      <div class="cd-end r"><b>Too many machines</b><span class="sub">tightly tuned, but expensive to run</span></div>
    </div>
    <div class="cd-sweet">↑ the sweet spot is the k that the silhouette score finds for you</div>
  </div>
</div>'''))

concept_diagram()

## The data

The control room hands you one table — **`waste_df`** — where every row is a **waste batch** logged
today. It carries some bookkeeping columns plus the physical properties of the waste and how much of
it there is (`quantity_kg`).

> Heads up: radioactivity and half-life span **many orders of magnitude** (from lab glassware to
> spent fuel). Keep that in mind — it will matter when we preprocess.


In [ ]:
import importlib
import hw3_data
importlib.reload(hw3_data)   # always pick up the latest version of the helper
waste_df = hw3_data.load_data()
waste_df.head()

In [ ]:
waste_df.describe()

## Part 1 — Visualize the data

Before any modelling, *look* at what you've got. The histograms below show the physical features
**exactly as they arrive** — raw, untouched. Don't worry that some look almost empty with one tall
spike on the left: that's a real property of this data, and **spotting it now is the whole point.**
Ask yourself: *which features are badly skewed, and do they all live on the same scale?* Keep that
observation in mind — in **Part 3** you'll learn a tool that fixes exactly this.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
NUM = hw3_data.PHYSICAL_FEATURES + hw3_data.REDUNDANT_FEATURES
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), NUM):
    vals = waste_df[col].to_numpy()
    # Raw values, no transformation — show the data as it really is.
    ax.hist(vals, bins=30, color=BLUE, edgecolor="white")
    ax.set_title(col, fontsize=9)
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))
axes.ravel()[-1].axis("off")
# chemical_class counts in the spare panel.
counts = waste_df["chemical_class"].value_counts()
axes.ravel()[-1].axis("on")
axes.ravel()[-1].bar(counts.index, counts.values, color=ORANGE, edgecolor="white")
axes.ravel()[-1].set_title("chemical_class", fontsize=9)
axes.ravel()[-1].tick_params(axis="x", rotation=30)
fig.suptitle("What the waste log looks like", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎯 Task 1 — a correlation heatmap

A heatmap of feature correlations tells us which columns move together. Build the correlation matrix
of the **numeric** columns and we'll plot it. Watch for any pair that lights up bright at **~1.0** —
that is two columns carrying the *same* information.

In [ ]:
# 🎯 Build a correlation matrix of the numeric physical columns (incl. the redundant one).
NUM = hw3_data.PHYSICAL_FEATURES + hw3_data.REDUNDANT_FEATURES
corr = ...   # 🎯 call .corr() on the numeric sub-frame

assert corr.shape == (len(NUM), len(NUM)), "corr should be a square matrix, one row/col per feature."
# The Bq and Ci radioactivity columns are the SAME quantity in different units → near-perfect corr.
assert corr.loc["radioactivity_Bq", "radioactivity_Ci"] > 0.99, \
    "Expected radioactivity_Bq and radioactivity_Ci to be almost perfectly correlated."
print("Correlation matrix ready ✅  (spot the ~1.0 pair!)")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.to_numpy(), cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUM))); ax.set_xticklabels(NUM, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(NUM))); ax.set_yticklabels(NUM, fontsize=8)
for i in range(len(NUM)):
    for j in range(len(NUM)):
        ax.text(j, i, f"{corr.to_numpy()[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(corr.to_numpy()[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Feature correlations — find the duplicate", fontweight="bold")
plt.tight_layout(); plt.show()

## Part 2 — Feature selection

Not every column helps a clustering. Two kinds of columns hurt us here:

1. **Bookkeeping / noise** — `batch_id`, `storage_shelf`, `barcode`. These are labels and random
   identifiers; they carry no information about *how to process* the waste. Worse, KMeans would
   happily treat a barcode number as a "big" feature.
2. **Redundant duplicates** — `radioactivity_Ci` is just `radioactivity_Bq` in other units (you saw
   the ~1.0 correlation). Keeping both **double-counts** radioactivity, silently giving it twice the
   weight in the distance calculation.

We also keep `quantity_kg` aside: it tells us *how much* waste there is (for the final report), not
*what kind* it is — so it shouldn't shape the clusters.

### 🎯 Task 2a — drop the bookkeeping columns

In [ ]:
# 🎯 Drop the identifier + noise columns. (hw3_data lists them for you.)
drop_cols = ...   # 🎯 combine the two lists (ID_COLS + NOISE_COLS)
features_df = ...   # 🎯 drop those columns from waste_df

for c in drop_cols:
    assert c not in features_df.columns, f"{c} should have been dropped."
print("Dropped:", drop_cols)
print("Columns left:", list(features_df.columns))

### 🎯 Task 2b — remove the redundant feature

Two columns with `|corr| ≈ 1` are the **same feature twice**. Keep `radioactivity_Bq`, drop its
curie twin so radioactivity counts once.

In [ ]:
# 🎯 Drop the redundant radioactivity column (keep the Bq one).
features_df = ...   # 🎯 drop the redundant radioactivity column (keep the Bq one)

# These are the columns we'll actually cluster on (everything except quantity).
CLUSTER_NUM = hw3_data.PHYSICAL_FEATURES         # the 6 physical numerics
CLUSTER_CAT = hw3_data.CATEGORICAL               # ['chemical_class']

assert "radioactivity_Ci" not in features_df.columns, "The redundant Ci column should be gone."
assert "radioactivity_Bq" in features_df.columns, "Keep the Bq radioactivity column."
assert set(CLUSTER_NUM + CLUSTER_CAT).issubset(features_df.columns)
print("Clustering features:", CLUSTER_NUM + CLUSTER_CAT)

## Part 3 — Preprocessing

KMeans groups points by **Euclidean distance**, so the *units* of each feature matter enormously.
Two problems to fix:

- **Wild scales / skew.** `radioactivity_Bq` runs from ~10⁶ to ~10¹², `half_life_years` from days to
  tens of thousands of years. Left raw, these dwarf everything else. We **log-transform** the heavy
  features (`hw3_data.LOG_FEATURES`) to tame them.
- **Different ranges.** Even after logging, watts and a 0–1 corrosiveness index live on different
  scales. We **standard-scale** every numeric feature to mean 0 / std 1 so each contributes fairly —
  this is the same `StandardScaler` standardization you applied in the **scikit-learn notebook**, so
  remember what you did there.
- **A categorical column.** `chemical_class` is text — we **one-hot encode** it into 0/1 columns.

A `ColumnTransformer` lets us apply the right transform to the right columns in one shot.

### 📖 New tool — the log transform

You haven't met this one yet, so let's unpack it. Some features don't vary by a *little* — they vary
by **multiplying factors**. Radioactivity goes from about **1,000,000 Bq** (lab glassware) to
**1,000,000,000,000 Bq** (spent fuel) — a **million-fold** jump. On a normal axis, all the small
batches get squashed into a thin sliver near zero while a few giants stretch the scale; you simply
can't *see* the structure, and KMeans can't either (one giant batch sits absurdly far from the rest).

The **log transform** replaces each value with its *exponent* — roughly, **"how many zeros"** it has:

| raw value | `log10(value)` |
|---|---|
| 1,000,000 (10⁶) | **6** |
| 100,000,000 (10⁸) | **8** |
| 1,000,000,000,000 (10¹²) | **12** |

So a million-fold spread (10⁶ → 10¹²) becomes a tidy span of just **6 → 12**. Equal *ratios* now
become equal *distances*: "10× bigger" always moves you the same step. That turns a lopsided,
unusable feature into an evenly-spread one — exactly what a distance-based method like KMeans needs.

Run the cell below to see it on our actual radioactivity column.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
raw = waste_df["radioactivity_Bq"].to_numpy()
logged = np.log10(raw)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))

# BEFORE — histogram of the raw values (in trillions of Bq for readable ticks).
ax1.hist(raw / 1e12, bins=60, color=RED, edgecolor="white")
ax1.set_title("BEFORE — raw radioactivity", fontweight="bold")
ax1.set_xlabel("radioactivity  (trillions of Bq)"); ax1.set_ylabel("number of batches")
ax1.text(0.5, 0.7,
         "Almost every batch falls in this\nfirst bar. The axis stretches out\nto 5 trillion for a handful of giants,\nso the rest is an unreadable spike.",
         transform=ax1.transAxes, fontsize=10, color="#7a2e2e",
         bbox=dict(boxstyle="round", fc="#fdecec", ec="#e3b7b7"))

# AFTER — histogram of log10(value): the same batches, now spread out.
ax2.hist(logged, bins=30, color=GREEN, edgecolor="white")
ax2.set_title("AFTER — log10(radioactivity)", fontweight="bold")
ax2.set_xlabel("exponent  (6 = a million,  9 = a billion,  12 = a trillion)")
ax2.set_ylabel("number of batches")
ax2.text(0.03, 0.7,
         "Now the batches spread across\nthe axis — you can finally see\nthe groups hiding in the data.",
         transform=ax2.transAxes, fontsize=10, color="#2d6a3e",
         bbox=dict(boxstyle="round", fc="#e9f5ee", ec="#a9d6ba"))

fig.suptitle("Same data, two rulers — the log transform un-squashes the distribution",
             fontweight="bold")
plt.tight_layout(); plt.show()

### 🎯 Task 3 — build the preprocessing as one reusable object

We'll fold **all three** steps — log the heavy columns, scale the numerics, one-hot the text — into a
single `ColumnTransformer` called `preprocess`. The heavy columns get a tiny two-step
`Pipeline` (`log10` **then** scale); the already-tame numerics just get scaled.

Why bundle it like this? Because once `preprocess` is **fitted**, we can reuse it on *any* future
waste — exactly what we'll need in Part 6 to handle new arrivals. Fill in the three `???`.

In [ ]:
# Heavy, orders-of-magnitude columns → log10 first, THEN scale.
LOG = hw3_data.LOG_FEATURES                            # radioactivity, half-life, heat
LIN = [c for c in CLUSTER_NUM if c not in LOG]         # density, corrosiveness, particle size

log_then_scale = Pipeline([
    ("log", ...),            # 🎯 the transform that tames orders of magnitude
    ("scale", StandardScaler()),
])

# 🎯 One transformer that routes each group of columns to the right treatment.
preprocess = ColumnTransformer([
    ("heavy", log_then_scale, LOG),
    ("tame", ..., LIN),                   # 🎯 scaler for the already-tame numerics
    ("cat", ..., CLUSTER_CAT),  # 🎯 encoder for the text col
])
X = np.asarray(preprocess.fit_transform(features_df))

# First 6 columns are numeric (3 heavy + 3 tame); they should be ~mean 0, ~std 1.
n_num = len(CLUSTER_NUM)
assert np.allclose(X[:, :n_num].mean(axis=0), 0, atol=1e-6), \
    "Numeric columns should be centred at 0 — did StandardScaler run? (check axis=0)"
assert np.allclose(X[:, :n_num].std(axis=0), 1, atol=1e-6), \
    "Numeric columns should have unit std."
print("Preprocessed matrix:", X.shape, "✅  (preprocess is now fitted and reusable)")

### The payoff — look at the features again

Remember the skewed mess from **Part 1**? Here are the **same physical features**, but now the heavy
ones (`radioactivity`, `half_life`, `heat`) are shown on the **log scale** you just applied. Compare
this to the Part 1 grid: the lopsided spikes have turned into spread-out, readable shapes — and you
can even start to see **separate humps**, which are the waste groups we're about to cluster.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, col in zip(axes.ravel(), hw3_data.PHYSICAL_FEATURES):
    vals = waste_df[col].to_numpy()
    if col in hw3_data.LOG_FEATURES:
        ax.hist(np.log10(vals), bins=30, color=GREEN, edgecolor="white")
        ax.set_title(f"log10({col})  ✅ logged", fontsize=9)
    else:
        ax.hist(vals, bins=30, color=BLUE, edgecolor="white")
        ax.set_title(f"{col}  (already tame)", fontsize=9)
    ax.set_ylabel("batches", fontsize=8)
fig.suptitle("After the log transform — the heavy features are finally readable "
             "(compare with Part 1)", fontweight="bold")
plt.tight_layout(); plt.show()

## Part 4 — Clustering: how many machines?

Each cluster will become **one machine setting**. So *"how many clusters?"* literally means *"how
many machines do we need tonight?"*. We don't know the answer up front, so we **search** over a range
of `k` and judge each with two tools:

- **Inertia** — *are the clusters tight?*
- **Silhouette score** — *are the clusters tight **and** well-separated?*

Both are new, so let's build the intuition before we use them.

### 📖 New tools — inertia & the silhouette score

**1. Inertia — "how spread out is each cluster?"**

Every cluster has a **centroid** (its centre point). Inertia is the **total squared distance from
every point to its own centroid** — add up all those little spokes and square them. A small inertia
means points sit snugly around their centre; a large inertia means loose, scattered clusters.

Here's the catch: inertia **always shrinks as you add more machines** (`k`). With one machine per
batch, inertia would hit 0 — but that's useless. So we don't look for the *minimum*; we look for the
**elbow**: the `k` after which adding another machine barely reduces inertia. That bend marks
"diminishing returns."

**2. Silhouette score — "are clusters tight *and* far apart?"**

Inertia only measures tightness; it ignores whether two clusters overlap. The **silhouette** fixes
that. For a single batch it compares two distances:

- **a** = average distance to the batches in **its own** cluster (cohesion — small is good),
- **b** = average distance to the batches in the **nearest other** cluster (separation — big is good).

$$ \text{silhouette} = \frac{b - a}{\max(a,\,b)} $$

- **≈ +1** → much closer to its own group than to any other → *perfectly placed* ✅
- **≈ 0** → sitting right on the border between two clusters 😐
- **< 0** → actually closer to a *different* cluster → *probably mis-assigned* ❌

We average it over all batches; the `k` with the **highest** silhouette is the cleanest split.
Run the cell below for the picture.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
def metric_intuition():
    rng = np.random.default_rng(3)

    def blob(cx, cy, n, s=11):
        return [(cx + rng.normal(0, s), cy + rng.normal(0, s)) for _ in range(n)]

    # ---- Panel A: inertia = squared spokes to each centroid ----
    cA1, cA2 = (62, 70), (165, 115)
    b1, b2 = blob(*cA1, 8), blob(*cA2, 8)
    spokes = ""
    for (cx, cy), pts, col in [(cA1, b1, BLUE), (cA2, b2, ORANGE)]:
        for x, y in pts:
            spokes += f'<line x1="{cx}" y1="{cy}" x2="{x:.0f}" y2="{y:.0f}" stroke="{col}" stroke-width="1.2" opacity="0.55"/>'
        spokes += "".join(f'<circle cx="{x:.0f}" cy="{y:.0f}" r="4" fill="{col}"/>' for x, y in pts)
    for (cx, cy) in (cA1, cA2):
        spokes += f'<path d="M{cx-5},{cy} h10 M{cx},{cy-5} v10" stroke="#333" stroke-width="2"/>'

    # ---- Panel B: silhouette = a (own) vs b (neighbour) for one point ----
    cB1, cB2 = (60, 95), (175, 95)
    own, nbr = blob(*cB1, 7), blob(*cB2, 7)
    px, py = 92, 92  # the highlighted point (belongs to the left cluster)
    bdots = "".join(f'<circle cx="{x:.0f}" cy="{y:.0f}" r="4" fill="{BLUE}" opacity="0.8"/>' for x, y in own)
    bdots += "".join(f'<circle cx="{x:.0f}" cy="{y:.0f}" r="4" fill="{ORANGE}" opacity="0.8"/>' for x, y in nbr)
    bdots += (f'<line x1="{px}" y1="{py}" x2="{cB1[0]}" y2="{cB1[1]}" stroke="{GREEN}" stroke-width="2.4"/>'
              f'<line x1="{px}" y1="{py}" x2="{cB2[0]}" y2="{cB2[1]}" stroke="{RED}" stroke-width="2.4" stroke-dasharray="5 3"/>'
              f'<text x="{(px+cB1[0])//2-4}" y="{py-8}" font-size="13" font-weight="700" fill="{GREEN}">a</text>'
              f'<text x="{(px+cB2[0])//2}" y="{py-8}" font-size="13" font-weight="700" fill="{RED}">b</text>'
              f'<circle cx="{px}" cy="{py}" r="6.5" fill="#fff" stroke="#222" stroke-width="2.5"/>')

    display(HTML(f'''
<style>
.mi{{font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f7f9ff,#f3fbf6);
    border:1px solid #e7eef0;border-radius:20px;padding:18px;margin:8px 0}}
.mi-row{{display:flex;gap:14px;flex-wrap:wrap;justify-content:center}}
.mi-card{{flex:1 1 300px;min-width:280px;background:#fff;border:1px solid #eef1f7;border-radius:14px;
    padding:12px 14px;box-shadow:0 4px 12px rgba(50,70,120,.05)}}
.mi-h{{font-weight:800;font-size:14px;color:#2c2350;margin-bottom:2px}}
.mi-sub{{font-size:11.5px;color:#8b86a6;margin-bottom:6px}}
.mi-note{{font-size:12px;color:#444;margin-top:6px;line-height:1.5}}
.mi-scale{{margin-top:8px;height:14px;border-radius:8px;
    background:linear-gradient(90deg,#c44e52,#e7c14a 50%,#55a868)}}
.mi-ends{{display:flex;justify-content:space-between;font-size:11px;color:#666;margin-top:3px}}
</style>
<div class="mi"><div class="mi-row">
  <div class="mi-card">
    <div class="mi-h">📏 Inertia — total squared spokes</div>
    <div class="mi-sub">distance from each point to its centroid (the ✛)</div>
    <svg viewBox="0 0 230 190" width="100%" height="180">{spokes}</svg>
    <div class="mi-note"><b>Smaller = tighter clusters.</b> It always drops as <i>k</i> grows, so we
    hunt the <b>elbow</b>, not the minimum.</div>
  </div>
  <div class="mi-card">
    <div class="mi-h">🎯 Silhouette — own vs nearest cluster</div>
    <div class="mi-sub">for one batch: a = own group, b = nearest other group</div>
    <svg viewBox="0 0 230 190" width="100%" height="180">{bdots}</svg>
    <div class="mi-note"><b>s = (b − a) / max(a, b).</b> Big <span style="color:#c44e52">b</span> and
    small <span style="color:#55a868">a</span> ⇒ s near +1 ✅</div>
    <div class="mi-scale"></div>
    <div class="mi-ends"><span>−1 mis-assigned</span><span>0 on the border</span><span>+1 ideal</span></div>
  </div>
</div></div>'''))

metric_intuition()

### 🎯 Task 4a — score every candidate k

For each `k`, fit KMeans and record its silhouette score and inertia. Fill the two `???`.

In [ ]:
K_RANGE = range(2, 9)
sil_scores, inertias = [], []
for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = ...                       # 🎯 fit and get a cluster label per row
    sil_scores.append(...)   # 🎯 score this clustering
    inertias.append(km.inertia_)
    print(f"k={k}:  silhouette={sil_scores[-1]:.3f}   inertia={inertias[-1]:.0f}")

assert len(sil_scores) == len(list(K_RANGE))
assert all(-1 <= s <= 1 for s in sil_scores), "Silhouette must be in [-1, 1]."
print("Scored every candidate k ✅")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
ks = list(K_RANGE)
best_idx = int(np.argmax(sil_scores))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(ks, sil_scores, "o-", color=GREEN, lw=2)
ax1.scatter([ks[best_idx]], [sil_scores[best_idx]], s=160, facecolors="none",
            edgecolors=RED, linewidths=2.5, zorder=5)
ax1.set_xlabel("k (number of machines)"); ax1.set_ylabel("silhouette score")
ax1.set_title("Higher is better → peak picks k", fontweight="bold")
ax2.plot(ks, inertias, "o-", color=BLUE, lw=2)
ax2.axvline(ks[best_idx], color=RED, ls="--", alpha=0.7)
ax2.set_xlabel("k (number of machines)"); ax2.set_ylabel("inertia (within-cluster spread)")
ax2.set_title("Look for the elbow", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎯 Task 4b — pick the best k

The silhouette curve has a clear peak. Pick the `k` that **maximises** it — that's your machine
count. (Hint: `np.argmax` gives the *index* into `sil_scores`; map it back to the actual `k` via
`K_RANGE`.)

In [ ]:
# 🎯 Find the k with the highest silhouette score.
best_idx = ...     # 🎯 index of the max silhouette
best_k = ...          # 🎯 the actual k at that index

assert best_k == hw3_data.N_TRUE_CLUSTERS, \
    f"Silhouette should peak at {hw3_data.N_TRUE_CLUSTERS} machines — got {best_k}."
print(f"✅ Optimal number of machines: {best_k}")

Now refit KMeans once at the chosen `k`, attach the cluster label to every batch, and look at
the result in 2-D (via PCA, which squashes our many features onto a plane just for plotting).

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
final_km = KMeans(n_clusters=best_k, n_init=10, random_state=0)
clusters = final_km.fit_predict(X)
features_df = features_df.copy()
features_df["machine"] = clusters     # which machine each batch is assigned to

pca = PCA(n_components=2, random_state=0).fit(X)   # keep it — we'll project new waste onto it later
coords = pca.transform(X)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for c in range(best_k):
    m = clusters == c
    ax1.scatter(coords[m, 0], coords[m, 1], s=22, alpha=0.75,
                color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)], label=f"Machine {c}")
ax1.set_title(f"{best_k} clusters in PCA space", fontweight="bold")
ax1.set_xlabel("PC 1"); ax1.set_ylabel("PC 2"); ax1.legend(fontsize=8)

sizes = pd.Series(clusters).value_counts().sort_index()
ax2.bar([f"M{c}" for c in sizes.index], sizes.values,
        color=[CLUSTER_COLORS[c % len(CLUSTER_COLORS)] for c in sizes.index], edgecolor="white")
ax2.set_title("Batches assigned per machine", fontweight="bold")
ax2.set_ylabel("number of batches")
for i, v in zip(range(len(sizes)), sizes.values):
    ax2.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

## Part 5 — Machines & their settings

You have your `k` machines. Now translate each cluster into a **machine configuration**: what kind of
waste does it handle, and how much of it? We summarise each machine by the **average physical
profile** of its batches and the **total mass** it must process tonight.

### 🎯 Task 5 — profile each machine

Group the batches by `machine` and compute (a) the **mean** of each physical feature and (b) the
**sum** of `quantity_kg`. Fill the two `???`.

In [ ]:
# 🎯 Average physical profile per machine.
profile = ...   # 🎯 mean physical profile per machine (group by "machine")

# 🎯 Total mass each machine has to process tonight.
load_kg = ...                 # 🎯 total quantity_kg per machine (group by "machine")

assert profile.shape == (best_k, len(hw3_data.PHYSICAL_FEATURES))
assert len(load_kg) == best_k
assert load_kg.sum() > 0
profile.assign(total_quantity_kg=load_kg.round(0)).round(2)

The heatmap below shows each machine's profile as **z-scores** (how far above/below average it
is on each feature) — that's what makes the machines visibly *different*. Then we turn each profile
into a concrete settings card.

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
# z-score the profile across machines so the contrasts pop.
z = (profile - profile.mean(axis=0)) / profile.std(axis=0)
fig, ax = plt.subplots(figsize=(9, 0.7 * best_k + 1.5))
im = ax.imshow(z.to_numpy(), cmap="coolwarm", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(hw3_data.PHYSICAL_FEATURES)))
ax.set_xticklabels(hw3_data.PHYSICAL_FEATURES, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(best_k)); ax.set_yticklabels([f"Machine {c}" for c in profile.index])
for i in range(best_k):
    for j in range(len(hw3_data.PHYSICAL_FEATURES)):
        ax.text(j, i, f"{z.to_numpy()[i, j]:+.1f}", ha="center", va="center", fontsize=7,
                color="white" if abs(z.to_numpy()[i, j]) > 1 else "black")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.03)
ax.set_title("Machine profiles (z-scores) — each row is a distinct setting", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
# Translate each machine's average profile into human-readable settings.
def setting(value, lo, hi):
    if value >= hi:   return "HIGH"
    if value <= lo:   return "LOW"
    return "MED"

rad = profile["radioactivity_Bq"]; heat = profile["heat_output_W"]; corr = profile["corrosiveness"]
cards = ""
for c in profile.index:
    shield = setting(rad[c], rad.quantile(0.33), rad.quantile(0.66))
    temp = setting(heat[c], heat.quantile(0.33), heat.quantile(0.66))
    liner = "corrosion-resistant" if corr[c] >= corr.median() else "standard"
    col = CLUSTER_COLORS[c % len(CLUSTER_COLORS)]
    cards += f'''<div style="flex:1 1 200px;border-left:6px solid {col};background:#fafbff;
      border-radius:10px;padding:12px 14px">
      <div style="font-weight:800;font-size:15px;color:#333">🛠️ Machine {c}</div>
      <div style="font-size:13px;color:#555;margin:6px 0 8px">{int(load_kg[c])} kg to process</div>
      <div style="font-size:12px;line-height:1.6">
        🛡️ Shielding: <b>{shield}</b><br>
        🌡️ Temperature: <b>{temp}</b><br>
        🧪 Liner: <b>{liner}</b><br>
        ⚙️ Half-life class: <b>{setting(profile["half_life_years"][c],
            profile["half_life_years"].quantile(0.33),
            profile["half_life_years"].quantile(0.66))}</b>
      </div></div>'''

display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#f5fff8,#f3f7ff);
     padding:18px;border-radius:18px">
  <h2 style="margin:0 0 4px">🌙 Tonight's machine plan</h2>
  <div style="font-size:14px;color:#555;margin-bottom:12px">
    <b>{best_k} machines</b> · {int(load_kg.sum())} kg of waste total</div>
  <div style="display:flex;gap:12px;flex-wrap:wrap">{cards}</div>
</div>'''))

## 📦 Package your workflow

You just ran the same recipe end-to-end: **select features → preprocess → search for k → cluster →
profile.** Tomorrow you'll want to run that *whole thing again* on a fresh day of waste — so let's
bundle the steps you wrote into one reusable function, `build_machine_plan(raw_df)`. (Read it: it's
literally your Parts 2–5, nothing new.) From now on, one call rebuilds an entire machine plan.

In [ ]:
def build_machine_plan(raw_df, k_range=range(2, 9)):
    '''Your Part 2-5 workflow, packaged: returns the machines fitted on `raw_df`.'''
    feats = raw_df[hw3_data.PHYSICAL_FEATURES + hw3_data.CATEGORICAL]   # feature selection (Part 2)

    prep = ColumnTransformer([                                          # preprocessing (Part 3)
        ("heavy", Pipeline([("log", FunctionTransformer(np.log10)), ("scale", StandardScaler())]),
         hw3_data.LOG_FEATURES),
        ("tame", StandardScaler(),
         [c for c in hw3_data.PHYSICAL_FEATURES if c not in hw3_data.LOG_FEATURES]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), hw3_data.CATEGORICAL),
    ])
    Xp = np.asarray(prep.fit_transform(feats))

    ks = list(k_range)                                                  # choose k by silhouette (Part 4)
    sils = [silhouette_score(Xp, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xp))
            for k in ks]
    k = ks[int(np.argmax(sils))]
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xp)

    out = raw_df.copy()
    out["machine"] = km.labels_                                         # profile (Part 5)
    return {"k": k, "prep": prep, "km": km, "pca": PCA(2, random_state=0).fit(Xp),
            "X": Xp, "data": out}

# quick check: re-running it on today's data recovers the same number of machines
_check = build_machine_plan(waste_df)
assert _check["k"] == best_k, "Packaged workflow should reproduce today's machine count."
print(f"build_machine_plan() ready ✅  (reproduced {best_k} machines on today's data)")

# Part 6 — Route new waste to its machine

It's the next morning and a **truck of fresh waste** pulls in. You don't re-think the whole plant for
every delivery — the machines are already set up. You just need to **send each new batch to the
machine that fits it**.

That's a one-liner with what you have: feed the new waste through your **fitted** `preprocess` and ask
`final_km` which cluster it lands in. Notice we use `predict` — **not** `fit_predict` — because we are
*reusing* the existing machines, not inventing new ones. (Same idea as the train-then-deploy split in
the scikit-learn notebook: you fit once, then apply to new data.)

In [ ]:
incoming = hw3_data.load_incoming()
incoming.head()

### 🎯 Task 6 — route every new batch

Chain your fitted `preprocess` and `final_km` into one `routing_pipeline`, then **predict** the
machine for each new batch. Fill the `???`.

In [ ]:
# Reuse the already-fitted pieces — no re-fitting!
routing_pipeline = Pipeline([("prep", preprocess), ("kmeans", final_km)])

incoming = incoming.copy()
incoming["machine"] = ...   # 🎯 predict (not fit_predict!) — reuse the machines

assert len(incoming["machine"]) == len(incoming)
assert incoming["machine"].between(0, best_k - 1).all(), "Every batch must land in an existing machine."
print("Routed", len(incoming), "new batches into the existing", best_k, "machines ✅")
incoming[["radioactivity_Bq", "half_life_years", "chemical_class", "quantity_kg", "machine"]]

In [ ]:
# 📦 Package your routing step as a reusable function for the control panel.
def route(plan, new_df):
    """Send each new batch to its nearest ALREADY-FITTED machine (no re-fitting)."""
    feats = new_df[hw3_data.PHYSICAL_FEATURES + hw3_data.CATEGORICAL]      # the features we clustered on
    routing = Pipeline([("prep", plan["prep"]), ("kmeans", plan["km"])])  # 🎯 reuse the fitted prep + kmeans
    return np.asarray(routing.predict(feats))                             # 🎯 predict — never fit_predict

# self-check: every routed id must point at a real machine
_demo_plan = build_machine_plan(waste_df)
_demo_ids = route(_demo_plan, hw3_data.load_incoming(verbose=False))
assert _demo_ids.min() >= 0 and _demo_ids.max() < _demo_plan["k"], "Routed ids must be valid machine ids."
print("route() ready ✅")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
# Show the new arrivals (★) landing among today's clusters (faded dots), in the SAME PCA space.
inc_coords = pca.transform(preprocess.transform(incoming))
base_coords = pca.transform(X)
fig, ax = plt.subplots(figsize=(7.5, 6))
for c in range(best_k):
    m = clusters == c
    ax.scatter(base_coords[m, 0], base_coords[m, 1], s=18, alpha=0.18,
               color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)])
for c in range(best_k):
    m = incoming["machine"].to_numpy() == c
    ax.scatter(inc_coords[m, 0], inc_coords[m, 1], s=240, marker="*",
               color=CLUSTER_COLORS[c % len(CLUSTER_COLORS)], edgecolors="black", linewidths=1.2,
               label=f"→ Machine {c}")
ax.set_title("New arrivals (★) routed into the existing machines", fontweight="bold")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Part 7 — Tomorrow's plan, with missing data

Real logs are messy. Tonight's intake sheet for **tomorrow** came in with **holes**: a scanner
dropped some `quantity_kg` readings, and the `critical` safety flag (does this batch need
special handling?) wasn't recorded for every batch. You can't plan machines on a table full of gaps.

Good news: you have **months of past daily logs** where those fields *are* complete. So you'll do what
data scientists do — **train a model on the complete history to predict the missing values**, fill the
gaps, and *then* reuse your `build_machine_plan` to set tomorrow's machines.

- missing **`quantity_kg`** → a number → **regression**
- missing **`critical`** → a yes/no → **classification**

(These are the supervised tools from the scikit-learn notebook — same `train_test_split`, `fit`,
`predict`, and metrics.)

### 7.1 — The complete history (our training data)

In [ ]:
history = hw3_data.load_history()
print("Critical-waste rate in history:", round(history[hw3_data.CRITICAL_COL].mean(), 2))
history.head()

### 7.2 — Tomorrow's intake, with gaps

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
next_day = hw3_data.load_next_day()
missing = next_day.isna().sum()
missing = missing[missing > 0]
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(missing.index, missing.values, color=RED, edgecolor="white")
ax.set_title("Missing values in tomorrow's intake", fontweight="bold")
ax.set_ylabel("missing cells")
for i, v in enumerate(missing.values):
    ax.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()
next_day.head()

### 🎯 Task 7a — fill the missing **quantities** (regression)

Train a `RandomForestRegressor` on the history to predict `quantity_kg` from the physical features,
check it on held-out data, then use it to fill the blank quantities in `next_day`. Fill the `???`.

In [ ]:
FEAT = hw3_data.PHYSICAL_FEATURES
Xtr, Xte, ytr, yte = train_test_split(history[FEAT], history[hw3_data.QUANTITY_COL],
                                      test_size=0.25, random_state=0)

reg = RandomForestRegressor(n_estimators=200, random_state=0)
reg.fit(...)                                  # 🎯 train reg on the TRAINING split (Xtr, ytr)
pred = reg.predict(Xte)
r2 = ...                            # 🎯 how much variance do we explain? (r2_score)
mae = mean_absolute_error(yte, pred)
print(f"Held-out R² = {r2:.2f}   MAE = {mae:.0f} kg")

# 🎯 Fill the missing quantities in next_day using the trained model.
gap = next_day[hw3_data.QUANTITY_COL].isna()
next_day.loc[gap, hw3_data.QUANTITY_COL] = ...   # 🎯 predict the missing quantities with reg

assert next_day[hw3_data.QUANTITY_COL].isna().sum() == 0, "There should be no missing quantities left."
assert r2 > 0.5, "The regressor should explain most of the variance — check the features/target."
print("Filled", int(gap.sum()), "missing quantities ✅")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(yte, pred, s=14, alpha=0.4, color=BLUE, edgecolors="none")
lo, hi = float(min(yte.min(), pred.min())), float(max(yte.max(), pred.max()))
ax.plot([lo, hi], [lo, hi], "--", color=RED, lw=2)
ax.set_xlabel("actual quantity_kg"); ax.set_ylabel("predicted quantity_kg")
ax.set_title(f"Regression check — R² = {r2:.2f}", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎯 Task 7b — fill the missing **critical flags** (classification)

Same idea, but the target is a yes/no, so we use a `RandomForestClassifier`. Train it, check accuracy,
then fill the missing `critical` flags. Fill the `???`.

In [ ]:
# Train only on the history rows where the flag is known.
known = history[hw3_data.CRITICAL_COL].notna()
Xtr, Xte, ytr, yte = train_test_split(history.loc[known, FEAT],
                                      history.loc[known, hw3_data.CRITICAL_COL].astype(int),
                                      test_size=0.25, random_state=0, stratify=history.loc[known, hw3_data.CRITICAL_COL])

clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(...)                                  # 🎯 train the classifier clf (Xtr, ytr)
pred = clf.predict(Xte)
acc = ...                    # 🎯 share predicted correctly (accuracy_score)
print(f"Accuracy = {acc:.2f}   precision = {precision_score(yte, pred):.2f}   recall = {recall_score(yte, pred):.2f}")

# 🎯 Fill the missing critical flags in next_day.
gap = next_day[hw3_data.CRITICAL_COL].isna()
next_day.loc[gap, hw3_data.CRITICAL_COL] = ...   # 🎯 predict the missing critical flags with clf
next_day[hw3_data.CRITICAL_COL] = next_day[hw3_data.CRITICAL_COL].astype(int)

assert next_day[hw3_data.CRITICAL_COL].isna().sum() == 0, "No missing critical flags should remain."
assert acc > 0.7, "The classifier should beat a coin flip comfortably."
print("Filled", int(gap.sum()), "missing critical flags ✅")

In [ ]:
# 📦 Package your gap-filling step (Part 7) as a reusable function for the control panel.
def fill_gaps(history_df, gappy_df):
    """Train on the COMPLETE history, then fill the missing cells of gappy_df (no NaNs left)."""
    feat = hw3_data.PHYSICAL_FEATURES
    out = gappy_df.copy()

    # missing quantities → RandomForest regression
    qcol = hw3_data.QUANTITY_COL
    reg = RandomForestRegressor(n_estimators=200, random_state=0)
    reg.fit(history_df[feat], history_df[qcol])                # 🎯 train on the full history
    qgap = out[qcol].isna()
    if qgap.any():
        out.loc[qgap, qcol] = reg.predict(out.loc[qgap, feat])  # 🎯 predict only the blank cells

    # missing critical-flags → RandomForest classification (train on the known rows only)
    ccol = hw3_data.CRITICAL_COL
    known = history_df[ccol].notna()
    clf = RandomForestClassifier(n_estimators=200, random_state=0)
    clf.fit(history_df.loc[known, feat], history_df.loc[known, ccol].astype(int))
    cgap = out[ccol].isna()
    if cgap.any():
        out.loc[cgap, ccol] = clf.predict(out.loc[cgap, feat])
    out[ccol] = out[ccol].astype(int)
    return out

# self-check: no gaps should remain after filling tomorrow's intake
_hist = hw3_data.load_history(verbose=False)
_filled = fill_gaps(_hist, hw3_data.load_next_day(verbose=False))
assert _filled[[hw3_data.QUANTITY_COL, hw3_data.CRITICAL_COL]].isna().sum().sum() == 0, "Fill every gap."
print("fill_gaps() ready ✅")

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
cm = confusion_matrix(yte, pred)
fig, ax = plt.subplots(figsize=(4.4, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["routine", "critical"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["routine", "critical"])
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=13, fontweight="bold")
ax.set_title("Critical-flag confusion matrix", fontweight="bold")
plt.tight_layout(); plt.show()

### 🎯 Task 7c — re-plan tomorrow's machines

`next_day` has **no gaps left**. Now just hand it to the function you packaged earlier — it reruns the
whole select → preprocess → cluster pipeline and hands back tomorrow's machines. Fill the `???`.

In [ ]:
assert next_day.isna().sum().sum() == 0, "Fill all gaps before planning!"
plan = ...        # 🎯 reuse your packaged Part 2-5 workflow (build_machine_plan on next_day)

print(f"Tomorrow's plan: {plan['k']} machines for {len(next_day)} batches.")
plan["data"]["machine"].value_counts().sort_index()

In [ ]:
#@title 📊 Visualization (double-click to view the code) { display-mode: "form" }
# Tomorrow's machine plan: settings + load + whether the machine handles critical waste.
nd = plan["data"]
prof = nd.groupby("machine")[hw3_data.PHYSICAL_FEATURES].mean()
load = nd.groupby("machine")[hw3_data.QUANTITY_COL].sum()
crit = nd.groupby("machine")[hw3_data.CRITICAL_COL].mean()   # fraction critical in each machine

def lvl(v, s):
    return "HIGH" if v >= s.quantile(0.66) else ("LOW" if v <= s.quantile(0.33) else "MED")

cards = ""
for c in prof.index:
    col = CLUSTER_COLORS[c % len(CLUSTER_COLORS)]
    danger = crit[c] >= 0.5
    badge = ('<span style="background:#c44e52;color:#fff;border-radius:6px;padding:1px 7px;'
             'font-size:11px">⚠️ CRITICAL</span>') if danger else \
            ('<span style="background:#55a868;color:#fff;border-radius:6px;padding:1px 7px;'
             'font-size:11px">routine</span>')
    cards += f'''<div style="flex:1 1 200px;border-left:6px solid {col};background:#fafbff;
      border-radius:10px;padding:12px 14px">
      <div style="font-weight:800;font-size:15px;color:#333">🛠️ Machine {c} {badge}</div>
      <div style="font-size:13px;color:#555;margin:6px 0 8px">{int(load[c])} kg to process</div>
      <div style="font-size:12px;line-height:1.6">
        🛡️ Shielding: <b>{lvl(prof["radioactivity_Bq"][c], prof["radioactivity_Bq"])}</b><br>
        🌡️ Temperature: <b>{lvl(prof["heat_output_W"][c], prof["heat_output_W"])}</b><br>
        ⚙️ Half-life class: <b>{lvl(prof["half_life_years"][c], prof["half_life_years"])}</b><br>
        🧪 Critical share: <b>{crit[c]*100:.0f}%</b>
      </div></div>'''

display(HTML(f'''
<div style="font-family:system-ui,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#fff7f0,#f3f7ff);
     padding:18px;border-radius:18px">
  <h2 style="margin:0 0 4px">🌅 Tomorrow's machine plan (from the repaired log)</h2>
  <div style="font-size:14px;color:#555;margin-bottom:12px">
    <b>{plan['k']} machines</b> · {int(load.sum())} kg forecast · gaps filled by your models</div>
  <div style="display:flex;gap:12px;flex-wrap:wrap">{cards}</div>
</div>'''))

## ✅ What you built

A full waste-operations pipeline — **unsupervised** clustering to *design* the machines, then
**supervised** models to *keep it running* when data goes missing:

**raw log → select features → preprocess → cluster (find k) → machine settings**
**→ route new arrivals → forecast missing data → re-plan tomorrow**

**Techniques practiced:** feature selection (correlation + dropping noise), log-transform +
`StandardScaler` + `OneHotEncoder` bundled in a reusable `ColumnTransformer` / `Pipeline`, `KMeans`
with `k` chosen by **silhouette** + the **inertia elbow**, `PCA` for 2-D views, `predict` vs
`fit_predict` to **route** new data into fitted clusters, and `RandomForest` **regression +
classification** (with `train_test_split`, R²/MAE, accuracy/precision/recall, confusion matrix) to
**impute missing values** before re-clustering.

**The one idea to remember:** the same fitted pipeline is an asset you **reuse** — to route new
waste, to repair a broken log, and to re-plan the next day — not something you rebuild from scratch
each time.

### Try it yourself
- Force `best_k = 3` or `7`. How do the profiles and the per-machine load change?
- Skip the log-transform in Part 3 — does the silhouette peak move? Why?
- In Part 7, raise `missing_frac` in `load_next_day(...)`. How far can the models stretch before the
  plan degrades?
- Swap the `RandomForest` models for a `LinearRegression` / `LogisticRegression`. Does the R² / accuracy hold up?

## 📦 Export your solution for the control panel

The course ends with a **Nuclear Central Manager** control panel (in `workshop/`) that runs your
homework as a live power-plant. Run the cell below to package this notebook's clustering workflow as
`solution_part3.py`, then drop that file into `workshop/solutions/`. The panel will route tonight's
waste with **your** machines instead of the built-in reference.

In [ ]:
# 📦 Export YOUR notebook as solution_part3.py for the Nuclear Central Manager control panel.
# This captures the *actual source* of the three functions you wrote above
# (fill_gaps, build_machine_plan, route) — not a canned answer.
import inspect, os

HEADER = """# Part 3 solution — waste-machine clustering.
# Exported straight from the HW3 notebook: the three functions below are YOUR code.
import os, sys

import numpy as np
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Make the course HW3 data helper importable from wherever this file is dropped.
for _rel in ("../homework/helpers", "../../workshop/homework/helpers",
             "helpers", "../../exercises", "../exercises", "exercises", "."):
    _cand = os.path.abspath(os.path.join(os.path.dirname(__file__), _rel))
    if os.path.exists(os.path.join(_cand, "hw3_data.py")):
        sys.path.insert(0, _cand)
        break
import hw3_data  # noqa: E402  (path injected just above)
"""

WRAPPER = """

class WasteSolution:
    def fill_gaps(self, history, gappy):
        return fill_gaps(history, gappy)

    def build_machine_plan(self, df, k_range=range(2, 9)):
        return build_machine_plan(df, k_range)

    def route(self, plan, new_df):
        return route(plan, new_df)


def get_solution():
    return WasteSolution()
"""

# Grab the three functions exactly as you defined them in this notebook.
try:
    bodies = [inspect.getsource(fn) for fn in (fill_gaps, build_machine_plan, route)]
except NameError as exc:
    raise NameError(
        "Run the cells that define fill_gaps, build_machine_plan and route "
        "(and re-run them after any edit) before exporting."
    ) from exc

source = HEADER + "\n" + "\n\n".join(b.rstrip() + "\n" for b in bodies) + WRAPPER

# Write next to the control panel (workshop/solutions/) when reachable, plus a local copy.
targets = []
panel = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "solutions", "solution_part3.py"))
if os.path.isdir(os.path.dirname(panel)):
    targets.append(panel)
targets.append(os.path.abspath(os.path.join(os.getcwd(), "solution_part3.py")))

for dest in dict.fromkeys(targets):  # de-dup while keeping order
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, "w") as f:
        f.write(source)
    print("Wrote", dest)

print("\nDrop solution_part3.py into workshop/solutions/ (already done if the path above is there),")
print("then relaunch the control panel — it will route tonight's waste with YOUR machines.")

try:  # in Colab, also offer the file as a download
    from google.colab import files
    files.download("solution_part3.py")
except Exception:
    pass